[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andreas-he/ai-safety-challenges/blob/main/challenges/2026-04-27-sae-reconstruction/challenge.ipynb)

# Day 11 — SAE Reconstruction: Implement the Sparse Autoencoder Core

**Category:** Mech Interp | **Format:** short-coding | **Difficulty:** standard (~25 min)

**Relevance:** Sparse Autoencoders (SAEs) are the primary tool in modern mech interp for decomposing transformer residual streams into interpretable features. Implementing the encode-decode-loss cycle is direct LASR Stage 3 prep, relevant to SAIGE's inoculation project (feature-level interventions), and foundational for tracking which features collapse as compute scales.

## Problem Summary

Implement three functions that together form the SAE training loop:
1. `sae_encode` — project activations into a sparse, high-dimensional feature space via ReLU
2. `sae_decode` — reconstruct the original activation from sparse features
3. `sae_loss` — compute reconstruction error + L1 sparsity penalty

## The Idea

A **Sparse Autoencoder (SAE)** learns a dictionary of features from transformer activations. Given a residual stream vector **x** ∈ ℝ^d_model, it:

1. **Encodes** to a high-dimensional, *sparse* feature space: **f** = ReLU(W_enc (x − b_pre) + b_enc)
2. **Decodes** back: **x̂** = W_dec **f** + b_pre
3. **Trains** on: *L* = ‖x − x̂‖₂² + λ ‖f‖₁

Two design choices encode key invariants:
- **b_pre symmetry:** subtracting before encoding and adding back after decoding centres the reconstruction around the data mean
- **Unit-norm decoder columns:** forces the decoder to commit to a specific *direction*; magnitude lives in the feature value alone

## Setup

In [ ]:
import numpy as np
np.random.seed(42)

d_model, n_features = 4, 8           # expansion factor = 2×

b_pre = np.array([-0.1,  0.05,  0.0, -0.05])   # mean-centre bias (d_model,)
W_enc = np.random.randn(n_features, d_model) * 0.5  # encoder weight (8, 4)
b_enc = np.zeros(n_features)                          # encoder bias  (8,)
_W    = np.random.randn(d_model, n_features)          # raw decoder
W_dec = _W / np.linalg.norm(_W, axis=0, keepdims=True)  # decoder (4, 8), unit-norm cols

lambda_l1 = 0.01
x = np.array([1.0, 0.5, -0.3, 0.8])

print(f"d_model={d_model}, n_features={n_features}")
print(f"W_enc shape: {W_enc.shape}, W_dec shape: {W_dec.shape}")
print(f"Decoder column norms (should all be 1.0): {np.linalg.norm(W_dec, axis=0).round(4)}")

## Task 1 — `sae_encode`

Compute the sparse feature activations from an input activation vector.

**Formula:** f = ReLU(W_enc @ (x − b_pre) + b_enc)

**Why b_pre?** We subtract the mean activation before encoding so the SAE learns residuals rather than fitting the mean — the decoder can then add b_pre back and reconstruct in the original space.

**Why ReLU?** Unlike smooth activations, ReLU creates *exact zeros* — a feature is fully off or partially on. This binary character is what makes SAE features interpretable.

In [ ]:
def sae_encode(x, W_enc, b_enc, b_pre):
    """
    Args:
        x:     activation vector, shape (d_model,)
        W_enc: encoder weight,    shape (n_features, d_model)
        b_enc: encoder bias,      shape (n_features,)
        b_pre: pre-encoder bias,  shape (d_model,)
    Returns:
        f: sparse feature activations, shape (n_features,), all >= 0
    """
    raise NotImplementedError

### Tests — Task 1

In [ ]:
f = sae_encode(x, W_enc, b_enc, b_pre)

assert f.shape == (n_features,), f"Expected shape ({n_features},), got {f.shape}"
assert np.all(f >= 0), "All feature activations must be >= 0 (ReLU)"

f_ref = np.maximum(0, W_enc @ (x - b_pre) + b_enc)
assert np.allclose(f, f_ref, atol=1e-6), f"Output mismatch\n  got:      {f}\n  expected: {f_ref}"

active = np.sum(f > 0)
print(f"✓ Task 1 passed — {active}/{n_features} features active, sparsity={1-active/n_features:.0%}")

## Task 2 — `sae_decode`

Reconstruct the original activation vector from sparse feature activations.

**Formula:** x̂ = W_dec @ f + b_pre

**Why add b_pre back?** The encoder subtracted b_pre from x before projecting. The decoder mirrors that: W_dec @ f captures the residual, then b_pre restores the mean. Without this, x̂ would live in a shifted coordinate frame and the loss would be systematically biased.

In [ ]:
def sae_decode(f, W_dec, b_pre):
    """
    Args:
        f:     sparse feature activations, shape (n_features,)
        W_dec: decoder weight,             shape (d_model, n_features)
        b_pre: pre-encoder bias,           shape (d_model,)
    Returns:
        x_hat: reconstructed activation, shape (d_model,)
    """
    raise NotImplementedError

### Tests — Task 2

In [ ]:
x_hat = sae_decode(f_ref, W_dec, b_pre)

assert x_hat.shape == (d_model,), f"Expected shape ({d_model},), got {x_hat.shape}"

x_hat_ref = W_dec @ f_ref + b_pre
assert np.allclose(x_hat, x_hat_ref, atol=1e-6), f"Output mismatch\n  got:      {x_hat}\n  expected: {x_hat_ref}"

recon_err = np.linalg.norm(x - x_hat)
print(f"✓ Task 2 passed — reconstruction error ‖x − x̂‖₂ = {recon_err:.4f}")

## Task 3 — `sae_loss`

Compute the SAE training loss: reconstruction error + sparsity penalty.

**Formula:** L = ‖x − x̂‖₂² + λ ‖f‖₁

**Why L1 and not L2?** L2 pushes all features toward zero but never forces them there. L1 creates *exact* zeros because its gradient is ±λ everywhere except 0 — any feature whose signal is weaker than λ gets driven to exactly 0. This is the mathematical foundation of SAE sparsity.

In [ ]:
def sae_loss(x, x_hat, f, lambda_l1):
    """
    Args:
        x:          original activation,       shape (d_model,)
        x_hat:      reconstructed activation,  shape (d_model,)
        f:          sparse feature values,     shape (n_features,)
        lambda_l1:  sparsity coefficient,      float
    Returns:
        loss: scalar float = L2_recon + lambda_l1 * L1_sparsity
    """
    raise NotImplementedError

### Tests — Task 3

In [ ]:
loss = sae_loss(x, x_hat_ref, f_ref, lambda_l1)

assert isinstance(loss, (float, np.floating)), f"Expected float, got {type(loss)}"

recon_ref    = float(np.sum((x - x_hat_ref) ** 2))
sparsity_ref = lambda_l1 * float(np.sum(np.abs(f_ref)))
expected     = recon_ref + sparsity_ref
assert np.isclose(loss, expected, atol=1e-6), f"Got {loss:.6f}, expected {expected:.6f}\n  recon={recon_ref:.6f}, sparsity_penalty={sparsity_ref:.6f}"

print(f"✓ Task 3 passed — loss={loss:.6f} (recon={recon_ref:.6f} + sparsity={sparsity_ref:.6f})")

## Reflection

The SAE is a 3-line implementation, but its power comes from the tension it resolves: transformers are *polysemantic* (neurons respond to multiple unrelated concepts) because d_model dimensions must encode vastly more than d_model features. The SAE projects into an *overcomplete* basis where each basis vector corresponds to one feature — the L1 penalty is the only thing preventing the model from using all features at once, which would collapse it back into polysemanticity.

📚 **Further reading:** *Features as Rewards* (Prasad, Watts, Merullo, Gala, Lewis, McGrath, Lubana — Goodfire, 2026) takes the exact encode→decode pipeline you just built and uses SAE feature activations as *reward signals* in RLHF — turning mech interp from a diagnostic tool into an active alignment technique. [Read at goodfire.ai/research/rlfr](https://www.goodfire.ai/research/rlfr)

<details>
<summary><b>Solution</b></summary>

```python
def sae_encode(x, W_enc, b_enc, b_pre):
    return np.maximum(0, W_enc @ (x - b_pre) + b_enc)

def sae_decode(f, W_dec, b_pre):
    return W_dec @ f + b_pre

def sae_loss(x, x_hat, f, lambda_l1):
    recon    = float(np.sum((x - x_hat) ** 2))
    sparsity = lambda_l1 * float(np.sum(np.abs(f)))
    return recon + sparsity
```

**Key insight:** Each function is 1–2 lines. The depth is in the design choices:

- **b_pre symmetry:** encoder subtracts, decoder adds back — reconstruction operates in the same coordinate frame as the original activation.
- **Unit-norm W_dec columns:** prevents magnitude-cheating (one huge decoder direction + tiny feature value = never sparse). The decoder encodes *direction*; feature value encodes *magnitude*.
- **ReLU → exact zeros:** genuine sparsity means features are either off (0) or on (positive). This binary character is what makes features interpretable as circuit components.
- **L1 vs L2 penalty:** L2 makes features small; L1 makes *some* features exactly 0, creating the sparse activation profile researchers use to identify monosemantic features.

The RLFR paper (Goodfire, 2026) extends this: instead of just diagnosing SAE features post-hoc, it uses them as reward signals in RLHF — the encode→decode pipeline becomes the critic in a reinforcement loop that directly optimises for feature-level behaviour.
</details>